In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

In [6]:
class SystemDataProcessor:
    def __init__(self, gen_csv_path, storage_csv_path=None):
        self.gen_csv_path = gen_csv_path
        self.storage_csv_path = storage_csv_path
        self.gen_data = None
        self.storage_data = None

    def load_data(self):
        try:
            self.gen_data = pd.read_csv(self.gen_csv_path).head(5)
            self.storage_data = pd.read_csv(self.storage_csv_path).head(5)
            print(f"Generator and Storage data loaded successfully. {len(self.gen_data)} generators and {len(self.storage_data)} storages found.")
            return True
        except Exception as e:
            print(f"Error loading data: {str(e)}")
            return False

    def process_gen_data(self):
        id_col = 'GEN UID'
        column_mapping = {
            # 'Ramp Rate MW/Min': '', 
            # 'Fuel Price $/MMBTU': '', 
            # 'VOM': '',
            'PMax MW': 'CAP',
            'Ramp Rate MW/Min': 'RR',
        }

        relevant_cols = list(column_mapping.keys())
        existing_cols = [col for col in relevant_cols if col in self.gen_data.columns]
        selected_cols = [id_col] + existing_cols
        gen_data_filtered = self.gen_data[selected_cols].copy()

        rename_dict = {col: column_mapping[col] for col in column_mapping if col in gen_data_filtered.columns}
        gen_data_filtered.rename(columns=rename_dict, inplace=True)
        gen_data_filtered.set_index(id_col, inplace=True)

        # if 'Fuel Price $/MMBTU' in gen_data_filtered.columns and 'HR_avg_0' in self.gen_data.columns:
        #     gen_data_filtered['VC'] = self.gen_data['Fuel Price $/MMBTU'] * self.gen_data['HR_avg_0'] / 1000.0
        #     if 'VOM' in gen_data_filtered.columns:
        #         gen_data_filtered['VC'] += gen_data_filtered['VOM']
        
        # if 'VC' in gen_data_filtered.columns:
        #     gen_data_filtered['VCUP'] = gen_data_filtered['VC'] * 1.5
        #     gen_data_filtered['VCDN'] = gen_data_filtered['VC'] * 0.5

        params = ['CAP', 'RR']
        gen_data_dict = {}

        for param in params:
            if param in gen_data_filtered.columns:
                param_dict = {}
                for gen_idx in gen_data_filtered.index:
                    param_dict[gen_idx] = gen_data_filtered.loc[gen_idx, param]
                gen_data_dict[param] = param_dict
            
        return gen_data_dict

    def process_storage_data(self):
        if self.storage_data is None or self.storage_data.empty:
            return {}
            
        try:
            id_col = 'GEN UID'

            column_mapping = {
                'Max Volume GWh': 'E_MAX',
                'Rating MVA': 'P_MAX',
                'Initial Volume GWh': 'E0',
                # 'Storage Roundtrip Efficiency': 'ETA'
            }

            relevant_cols = list(column_mapping.keys())
            existing_cols = [col for col in relevant_cols if col in self.storage_data.columns]
                
            selected_cols = [id_col] + existing_cols
            storage_data_filtered = self.storage_data[selected_cols].copy()

            rename_dict = {col: column_mapping[col] for col in column_mapping if col in storage_data_filtered.columns}
            storage_data_filtered.rename(columns=rename_dict, inplace=True)    
            storage_data_filtered.set_index(id_col, inplace=True)

            storage_data_dict = {}

            if 'E_MAX' in storage_data_filtered.columns:
                storage_data_dict['E_MAX'] = {
                    storage_idx: float(storage_data_filtered.at[storage_idx, 'E_MAX']) * 1000.0  # GWh to MWh
                    for storage_idx in storage_data_filtered.index
                }
                
            if 'P_MAX' in storage_data_filtered.columns:
                storage_data_dict['P_MAX'] = {
                    storage_idx: float(storage_data_filtered.loc[storage_idx, 'P_MAX'])
                    for storage_idx in storage_data_filtered.index
                }
                
            # if 'E0' in storage_data_filtered.columns:
            #     storage_data_dict['E0'] = {
            #         storage_idx: float(storage_data_filtered.loc[old_idx, 'E0']) * 1000.0  # GWh to MWh
            #         for old_idx, storage_idx in storage_indices.items()
            #     }
                
            # if 'ETA' in storage_data_filtered.columns:
            #     for old_idx, storage_idx in storage_indices.items():
            #         eta_rt = float(storage_data_filtered.loc[old_idx, 'ETA']) / 100.0 if storage_data_filtered.loc[old_idx, 'ETA'] else 0.9
            #         eta_each = eta_rt ** 0.5
                    
            #         if 'ETA_CH' not in storage_data_dict:
            #             storage_data_dict['ETA_CH'] = {}
            #         if 'ETA_DCH' not in storage_data_dict:
            #             storage_data_dict['ETA_DCH'] = {}
                        
            #         storage_data_dict['ETA_CH'][storage_idx] = eta_each
            #         storage_data_dict['ETA_DCH'][storage_idx] = eta_each
                
            return storage_data_dict
            
        except Exception as e:
            print(f"Error processing storage data: {str(e)}")
            return {}
    
    def prepare_pyomo_data(self, num_periods=24, num_scenarios=5, num_tiers=4):
        """Prepare the data for the Pyomo model."""
        if self.gen_data is None:
            success = self.load_data()
            if not success:
                return None
                
        gen_data_dict = self.process_gen_data()
        storage_data_dict = self.process_storage_data()
        
        num_generators = len(gen_data_dict.get('CAP', {}))
        num_storage = len(storage_data_dict.get('E_MAX', {}))
        
        sets = {
            'T': {None: list(range(1, num_periods + 1))},
            'S': {None: list(range(1, num_scenarios + 1))},
            'G': {None: list(range(1, num_generators + 1))},
            'R': {None: list(range(1, num_tiers + 1))},
            'B': {None: list(range(1, num_storage + 1))}
        }
        
        demand_data = {}
        reda_data = {}
        
        re_scenarios = {}
        
        fo_params = {
            'D1': {None: 10},
            'D2': {None: 0.1},
            'PEN': {None: 500},
            'PENDN': {None: 200},
            'smallM': {None: 0.0001},
            'probTU': {r: 0.2 + 0.1 * (r-1) for r in range(1, num_tiers + 1)},
            'probTD': {r: 0.2 + 0.1 * (r-1) for r in range(1, num_tiers + 1)}
        }
        
        pyomo_data = {}
        # pyomo_data.update(sets)
        pyomo_data.update(gen_data_dict)
        pyomo_data.update(storage_data_dict)
        # pyomo_data.update(fo_params)
        # pyomo_data['DEMAND'] = demand_data
        # pyomo_data['REDA'] = reda_data
        # pyomo_data['RE'] = re_scenarios
        
        # TODO: UPDATE REQUIRED PARAMETERS
        required_params = [
            'CAP', 'VC', 'VCUP', 'VCDN', 'RR',
            'E_MAX', 'P_MAX', 'ETA_CH', 'ETA_DCH', 'E0', 'E_FINAL', 'STORAGE_COST',
            'D1', 'D2', 'PEN', 'PENDN', 'smallM', 'probTU', 'probTD',
            'DEMAND', 'REDA', 'RE'
        ]
        
        missing_params = [param for param in required_params if param not in pyomo_data]
        if missing_params:
            print(f"Warning: Missing parameters: {missing_params}")
        
        return pyomo_data

In [7]:
import json

gen_csv_path = 'system_data/gen.csv'
storage_csv_path = 'system_data/storage.csv'

system_data = SystemDataProcessor(gen_csv_path, storage_csv_path)
pyomo_system_data = system_data.prepare_pyomo_data()

# print(pyomo_system_data)
print(json.dumps(pyomo_system_data, indent=4, sort_keys=True))

# if pyomo_data:
#     print("\nProcessed data summary:")
#     print(f"Number of generators: {len(pyomo_data.get('G', {}).get(None, []))}")
#     print(f"Number of storage units: {len(pyomo_data.get('B', {}).get(None, []))}")
#     print(f"Number of time periods: {len(pyomo_data.get('T', {}).get(None, []))}")
#     print(f"Number of scenarios: {len(pyomo_data.get('S', {}).get(None, []))}")
    
#     # Print sample of generator capacities
#     if 'CAP' in pyomo_data:
#         print("\nGenerator capacities (sample):")
#         for i, (g, cap) in enumerate(pyomo_data['CAP'].items()):
#             print(f"Generator {g}: {cap} MW")
#             if i >= 4:  # Show first 5 generators
#                 break
    
#     # Print sample of storage capacities
#     if 'E_MAX' in pyomo_data:
#         print("\nStorage capacities (sample):")
#         for i, (b, cap) in enumerate(pyomo_data['E_MAX'].items()):
#             print(f"Storage {b}: {cap} MWh")
#             if i >= 2:  # Show first 3 storage units
#                 break
    
#     # Print sample of demand data
#     if 'DEMAND' in pyomo_data:
#         print("\nDemand data (sample):")
#         for i, (t, demand) in enumerate(pyomo_data['DEMAND'].items()):
#             print(f"Period {t}: {demand} MW")
#             if i >= 4:  # Show first 5 periods
#                 break
# else:
#     print("Failed to prepare Pyomo data.")

Generator and Storage data loaded successfully. 5 generators and 5 storages found.
{
    "CAP": {
        "101_CT_1": 20.0,
        "101_CT_2": 20.0,
        "101_STEAM_3": 76.0,
        "101_STEAM_4": 76.0,
        "102_CT_1": 20.0
    },
    "E_MAX": {
        "122_HYDRO_1": 1000.0,
        "122_HYDRO_2": 1000.0,
        "212_CSP_1": 1200.0,
        "313_STORAGE_1": 150.0,
        "313_STORAGE_2": 150.0
    },
    "P_MAX": {
        "122_HYDRO_1": 50.0,
        "122_HYDRO_2": 50.0,
        "212_CSP_1": 200.0,
        "313_STORAGE_1": 50.0,
        "313_STORAGE_2": 50.0
    },
    "RR": {
        "101_CT_1": 3.0,
        "101_CT_2": 3.0,
        "101_STEAM_3": 2.0,
        "101_STEAM_4": 2.0,
        "102_CT_1": 3.0
    }
}


In [5]:
import json

gen_data = pd.read_csv('system_data/gen.csv').head(5)

id_col = 'GEN UID'
column_mapping = {
    # 'Ramp Rate MW/Min': '', 
    # 'Fuel Price $/MMBTU': '', 
    # 'VOM': '',
    'PMax MW': 'CAP',
    'Ramp Rate MW/Min': 'RR',
}

relevant_cols = list(column_mapping.keys())
existing_cols = [col for col in relevant_cols if col in gen_data.columns]
selected_cols = [id_col] + existing_cols
gen_data_filtered = gen_data[selected_cols].copy()

rename_dict = {col: column_mapping[col] for col in column_mapping if col in gen_data_filtered.columns}
gen_data_filtered.rename(columns=rename_dict, inplace=True)
gen_data_filtered.set_index(id_col, inplace=True)

print(gen_data_filtered)

params = ['CAP', 'RR']
gen_data_dict = {}

for param in params:
    if param in gen_data_filtered.columns:
        param_dict = {}
        for gen_idx in gen_data_filtered.index:
            param_dict[gen_idx] = gen_data_filtered.loc[gen_idx, param]
        gen_data_dict[param] = param_dict

# print(gen_data_dict)

print(json.dumps(gen_data_dict, indent=4, sort_keys=True))
# if 'Fuel Price $/MMBTU' in gen_data_filtered.columns and 'HR_avg_0' in self.gen_data.columns:
#     gen_data_filtered['VC'] = self.gen_data['Fuel Price $/MMBTU'] * self.gen_data['HR_avg_0'] / 1000.0
#     if 'VOM' in gen_data_filtered.columns:
#         gen_data_filtered['VC'] += gen_data_filtered['VOM']

# if 'VC' in gen_data_filtered.columns:
#     gen_data_filtered['VCUP'] = gen_data_filtered['VC'] * 1.5
#     gen_data_filtered['VCDN'] = gen_data_filtered['VC'] * 0.5

# gen_data_filtered.set_index(id_col, inplace=True)



# gen_indices = {old_idx: i+1 for i, old_idx in enumerate(gen_data_filtered.index)}



              CAP   RR
GEN UID               
101_CT_1     20.0  3.0
101_CT_2     20.0  3.0
101_STEAM_3  76.0  2.0
101_STEAM_4  76.0  2.0
102_CT_1     20.0  3.0
{
    "CAP": {
        "101_CT_1": 20.0,
        "101_CT_2": 20.0,
        "101_STEAM_3": 76.0,
        "101_STEAM_4": 76.0,
        "102_CT_1": 20.0
    },
    "RR": {
        "101_CT_1": 3.0,
        "101_CT_2": 3.0,
        "101_STEAM_3": 2.0,
        "101_STEAM_4": 2.0,
        "102_CT_1": 3.0
    }
}
